In [3]:
pip install langchain-google-vertexai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pymongo

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_google_vertexai import VertexAIEmbeddings

PROJECT_ID = 'qwiklabs-gcp-02-32e2d3dbe018'
REGION = 'europe-west1'

embeddings = VertexAIEmbeddings(
    model_name="text-embedding-004",
    project=PROJECT_ID,
    location=REGION
)


/tmp/ipykernel_9113/4211266875.py:6: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embeddings = VertexAIEmbeddings(
/tmp/ipykernel_9113/4211266875.py:6: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  embeddings = VertexAIEmbeddings(


In [6]:
pip install "pymongo[srv]"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
MONGODB_URI = ''

In [8]:
pip install langchain-mongodb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain_mongodb import MongoDBAtlasVectorSearch
from pymongo import MongoClient

client = MongoClient(MONGODB_URI)
db = client["data_quality_rag"]

examples_vs = MongoDBAtlasVectorSearch(
    collection=db["examples"],
    embedding=embeddings,
    index_name="examples_vector_index"
)

metadata_vs = MongoDBAtlasVectorSearch(
    collection=db["metadata"],
    embedding=embeddings,
    index_name="metadata_vector_index"
)


In [10]:
pip install langchain

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install langchain-core

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [12]:
from langchain_core.documents import Document
import json

docs = []
with open("examples.json") as f:
    for spec in json.load(f):
        text = f"""
        Type: {spec['rule_type']}
        Table: {spec['table']}
        Column: {spec['column']}
        Description: {spec['description']}
        Filter: {spec['filter']}
        Values: {spec['values']}
        """
        docs.append(Document(page_content=text, metadata=spec))

examples_vs.add_documents(docs)


['697321c05d2025410025108e']

In [13]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd
from langchain_core.documents import Document

df = pd.read_csv("metadata.csv")

docs = []
for _, row in df.iterrows():
    text = f"""
    Table: {row['table']}
    Column: {row['column']}
    Description: {row['description']}
    Data type: {row['data_type']}
    Allowed values: {row['allowed_values']}
    """
    docs.append(Document(page_content=text, metadata=row.to_dict()))

metadata_vs.add_documents(docs)


['697321c65d2025410025108f', '697321c65d20254100251090']

In [15]:
def retrieve_context(rule: dict, k=3):
    query = f"""
    Rule type: {rule['rule_type']}
    Table: {rule['table']}
    Column: {rule['column']}
    Description: {rule['description']}
    """

    examples = examples_vs.similarity_search(query, k=k)
    metadata = metadata_vs.similarity_search(query, k=k)

    return {
        "examples": examples,
        "metadata": metadata
    }


In [16]:
def build_rag_prompt(rule, schema, rag_context):
    examples_text = "\n".join(
        [doc.page_content for doc in rag_context["examples"]]
    )

    metadata_text = "\n".join(
        [doc.page_content for doc in rag_context["metadata"]]
    )

    return f"""
Tu es un expert senior en Data Quality et SQL.

### CONTEXTE MÉTIER
#### Exemples de spécifications similaires
{examples_text}

#### Documentation des données
{metadata_text}

### Règle fonctionnelle
Code règle : {rule['rule_code']}
Type : {rule['rule_type']}
Table : {rule['table']}
Colonne : {rule['column']}
Description : {rule['description']}

### Schéma
{schema}

### CONTRAINTES ABSOLUES
- Réponse UNIQUEMENT en JSON valide
- Valeurs SQL entre doubles quotes
- Aucun texte hors JSON

### JSON ATTENDU
{{
  "rule_code": "",
  "rule_type": "",
  "table": "",
  "column": "",
  "description": "",
  "filter": "",
  "values": ""
}}
"""


In [19]:
pip install google-cloud-aiplatform

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [20]:
from google.cloud import aiplatform
from vertexai.preview.generative_models import GenerativeModel
import json
import re

# Initialisation Vertex AI
aiplatform.init(project=PROJECT_ID, location=REGION)

model = GenerativeModel("gemini-2.5-flash")

/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [22]:
def generate_rule_spec_rag(rule):
    rag_context = retrieve_context(rule)

    prompt = build_rag_prompt(
        rule=rule,
        schema=rule["schema"],
        rag_context=rag_context
    )

    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": 0.0,
            "max_output_tokens": 1024,
            "response_mime_type": "application/json"
        }
    )

    return json.loads(response.text)


In [23]:
rule_input = {
    "rule_code": "REG0001",
    "rule_type": "completeness",
    "table": "client",
    "column": "Brand",
    "description": "La colonne (Brand) doit être remplie sauf si le client est un particulier",
    "schema": """
    client(
      IdClient INT,
      NomClient STRING,
      Brand STRING,
      TypeClient STRING
    )
    """
}

spec = generate_rule_spec_rag(rule_input)

print(json.dumps(spec, indent=2, ensure_ascii=False))

{
  "rule_code": "REG0001",
  "rule_type": "completeness",
  "table": "client",
  "column": "Brand",
  "description": "La colonne (Brand) doit être remplie sauf si le client est un particulier",
  "filter": "TypeClient != 'particulier'",
  "values": "Brand IS NOT NULL"
}
